In [106]:

from collections.abc import Iterable


class Problem[S, A]:
    def __init__(self, initial_state, goal_state) -> None:
        self.initial_state = initial_state
        self.goal_state = goal_state

    def get_actions(self, state: S) -> Iterable[S]:
        raise NotImplementedError

    def apply_action(self, state: S, action: A) -> S:
        raise NotImplementedError

    def goal_test(self, state: S) -> bool:
        return self.goal_state == state

    def action_cost(self, state1: S, action: A, state2: S) -> float:
        return 1

    def h(self, state: S) -> float:
        return 0.0

In [107]:
class GraphAStarProblem(Problem[str, str]):
  def __init__(
      self, initial: str, goal: str, graph: dict[str, dict[str, float]]
  ) -> None:
    super().__init__(initial, goal)
    self.graph: dict[str, dict[str, float]] = graph

  def get_actions(self, state: str) -> list[str]:
    return list(self.graph[state].keys())

  def apply_action(self, state: str, action: str) -> str:
    return action

  def action_cost(self, state1: str, action: str, state2: str) -> float:
    return self.graph[state1][state2]

  def h(self, state: str) -> float:
    return float(straight_line_distance[state])

In [108]:
distances_of_cities_in_romania: dict[str, dict[str, int]] = {
    "Arad": {"Zerind": 75, "Sibiu": 140, "Timisoara": 118},
    "Zerind": {"Arad": 75, "Oradea": 71},
    "Oradea": {"Zerind": 71, "Sibiu": 151},
    "Sibiu": {"Arad": 140, "Oradea": 151, "Fagaras": 99, "Rimnicu Vilcea": 80},
    "Timisoara": {"Arad": 118, "Lugoj": 111},
    "Lugoj": {"Timisoara": 111, "Mehadia": 70},
    "Mehadia": {"Lugoj": 70, "Dobreta": 75},
    "Dobreta": {"Mehadia": 75, "Craiova": 120},
    "Craiova": {"Dobreta": 120, "Rimnicu Vilcea": 146, "Pitesti": 138},
    "Rimnicu Vilcea": {"Sibiu": 80, "Craiova": 146, "Pitesti": 97},
    "Fagaras": {"Sibiu": 99, "Bucarest": 211},
    "Pitesti": {"Rimnicu Vilcea": 97, "Craiova": 138, "Bucarest": 101},
    "Bucarest": {"Fagaras": 211, "Pitesti": 101, "Giurgiu": 90, "Urziceni": 85},
    "Giurgiu": {"Bucarest": 90},
    "Urziceni": {"Bucarest": 85, "Hirsova": 98, "Vaslui": 142},
    "Hirsova": {"Urziceni": 98, "Eforie": 86},
    "Eforie": {"Hirsova": 86},
    "Vaslui": {"Urziceni": 142, "Iasi": 92},
    "Iasi": {"Vaslui": 92, "Neamt": 87},
    "Neamt": {"Iasi": 87},
}

straight_line_distance = {
    'Arad': 366,
    'Bucarest': 0,
    'Craiova': 160,
    'Dobreta': 242,
    'Eforie': 161,
    'Fagaras': 178,
    'Giurgiu': 77,
    'Hirsova': 151,
    'Iasi': 226,
    'Lugoj': 244,
    'Mehadia': 241,
    'Neamt': 234,
    'Oradea': 380,
    'Pitesti': 98,
    'Rimnicu Vilcea': 193,
    'Sibiu': 253,
    'Timisoara': 329,
    'Urziceni': 80,
    'Vaslui': 199,
    'Zerind': 374,
}

In [109]:
class Node[S, A]:
    def __init__(
        self,
        state: S,
        parent: "Node[S, A] | None" = None,
        action: A | None = None,
        path_cost: float = 0,
    ) -> None:
        self.state = state
        self.parent = parent
        self.action = action
        self.path_cost = path_cost

    def path(self) -> list[S]:
        path_list: list[Node[S]] = []

        node: Node = self
        while node:
            path_list.append(node.state)
            node: Node = node.parent

        path_list.reverse()

        return path_list

    def child_node(self, problem: GraphAStarProblem, action: A) -> "Node[S]":
        next_state = problem.apply_action(self.state, action)
        step_cost = problem.action_cost(self.state, action, next_state)

        return Node(next_state, self, action, self.path_cost + step_cost)

    def expand(self, problem: GraphAStarProblem) -> "list[Node[S]]":
        return [
            self.child_node(problem, action)
            for action in problem.get_actions(self.state)
        ]

In [110]:
def hill_climbing(
    problem: GraphAStarProblem[str, str],
) -> tuple[Node[str, str], bool]:
  current_node: Node[str, str] = Node(problem.initial_state)

  while True:
    if problem.goal_test(current_node.state):
      return current_node, True

    neighbors: list[Node[str, str]] = current_node.expand(problem)
    if not neighbors:
      break

    best_neighbor: Node[str, str] = min(
        neighbors, key=lambda child: problem.h(child.state)
    )

    if problem.h(best_neighbor.state) >= problem.h(current_node.state):
      break

    current_node = best_neighbor

  return current_node, problem.is_goal(current_node.state)

In [111]:
arad_to_bucarest_problem: GraphAStarProblem[str, str] = GraphAStarProblem(
    "Arad", "Bucarest", distances_of_cities_in_romania
)

In [112]:
(hill_climb_result, reached_goal) = hill_climbing(arad_to_bucarest_problem)
node_path = hill_climb_result.path()

print(" -> ".join(node_path))

Arad -> Sibiu -> Fagaras -> Bucarest
